<a href="https://colab.research.google.com/github/Dartrint/GuppyLM/blob/main/GuppyLM_Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GuppyLM — Your Friendly Fish

Train a ~9M parameter LLM that talks like a small fish.

**What this notebook does:**
1. Downloads 60K fish conversation dataset from HuggingFace
2. Trains a BPE tokenizer on the data
3. Trains a 6-layer vanilla transformer (8.7M params)
4. Tests the model with sample conversations

**Architecture:** 6 layers, 384 dim, 6 heads, ReLU FFN, LayerNorm, 4096 vocab

**Runtime:** ~5 min on T4 GPU

**Result:** A fish that speaks in short lowercase sentences about water, food, and light.

## 1. Setup

Install dependencies and create a clean working directory.

In [1]:
!pip install -q torch tokenizers tqdm numpy datasets huggingface_hub

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch 2.10.0+cu128
CUDA: True
GPU: Tesla T4


In [2]:
import os, shutil

# Start fresh — removes stale files from previous runs
if os.path.exists('/content/guppy'):
    shutil.rmtree('/content/guppy')
os.makedirs('/content/guppy')
os.chdir('/content/guppy')
print(f'Working dir: {os.getcwd()}')

Working dir: /content/guppy


## 2. Source Files

Write the model code to disk. These are the only files needed:
- `config.py` — model and training hyperparameters
- `model.py` — transformer architecture
- `dataset.py` — data loading and batching
- `train.py` — training loop
- `inference.py` — chat interface

In [3]:
%%writefile config.py
"""GuppyLM configuration."""

from dataclasses import dataclass


@dataclass
class GuppyConfig:
    vocab_size: int = 4096
    max_seq_len: int = 128
    d_model: int = 384
    n_layers: int = 6
    n_heads: int = 6
    ffn_hidden: int = 768
    dropout: float = 0.1

    # Special tokens
    pad_id: int = 0
    bos_id: int = 1           # <|im_start|>
    eos_id: int = 2           # <|im_end|>


@dataclass
class TrainConfig:
    batch_size: int = 32
    learning_rate: float = 3e-4
    min_lr: float = 3e-5
    weight_decay: float = 0.1
    warmup_steps: int = 200
    max_steps: int = 10000
    eval_interval: int = 200
    save_interval: int = 500
    grad_clip: float = 1.0
    device: str = "auto"
    seed: int = 42
    data_dir: str = "data"
    output_dir: str = "checkpoints"


Writing config.py


In [4]:
%%writefile model.py
"""
GuppyLM — a tiny fish brain.

Vanilla transformer: multi-head attention, ReLU FFN, LayerNorm, learned positional embeddings.
No GQA, no SwiGLU, no parallel residual, no RoPE. As simple as it gets.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from config import GuppyConfig


class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads

        self.qkv = nn.Linear(config.d_model, 3 * config.d_model)
        self.out = nn.Linear(config.d_model, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float("-inf"))
        attn = self.dropout(F.softmax(attn, dim=-1))
        return self.out((attn @ v).transpose(1, 2).contiguous().view(B, T, C))


class FFN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.up = nn.Linear(config.d_model, config.ffn_hidden)
        self.down = nn.Linear(config.ffn_hidden, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.down(F.relu(self.up(x))))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.d_model)
        self.attn = Attention(config)
        self.norm2 = nn.LayerNorm(config.d_model)
        self.ffn = FFN(config)

    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x


class GuppyLM(nn.Module):
    def __init__(self, config: GuppyConfig):
        super().__init__()
        self.config = config

        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.max_seq_len, config.d_model)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
        self.norm = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # tie weights

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        mask = torch.tril(torch.ones(T, T, device=idx.device)).unsqueeze(0).unsqueeze(0)

        for block in self.blocks:
            x = block(x, mask)

        logits = self.lm_head(self.norm(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.config.vocab_size),
                targets.view(-1),
                ignore_index=0,
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=64, temperature=0.7, top_k=50, **kwargs):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
            if next_id.item() == self.config.eos_id:
                break
        return idx, []

    def param_count(self):
        total = sum(p.numel() for p in self.parameters())
        return total, 0

    def param_summary(self):
        total, _ = self.param_count()
        return f"GuppyLM: {total:,} params ({total/1e6:.1f}M)"


Writing model.py


In [5]:
%%writefile dataset.py
"""GuppyLM dataset loading."""

import json

import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer


class GuppyDataset(Dataset):
    def __init__(self, path: str, tokenizer_path: str, max_len: int = 512):
        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.max_len = max_len
        self.samples = []

        with open(path) as f:
            for line in f:
                data = json.loads(line)
                ids = self.tokenizer.encode(data["text"]).ids
                if len(ids) > max_len:
                    ids = ids[:max_len]
                if len(ids) >= 2:
                    self.samples.append(ids)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]
        x = ids[:-1]
        y = ids[1:]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


def collate_fn(batch, pad_id=0):
    xs, ys = zip(*batch)
    max_len = max(len(x) for x in xs)
    padded_x = torch.full((len(xs), max_len), pad_id, dtype=torch.long)
    padded_y = torch.full((len(ys), max_len), pad_id, dtype=torch.long)
    for i, (x, y) in enumerate(zip(xs, ys)):
        padded_x[i, :len(x)] = x
        padded_y[i, :len(y)] = y
    return padded_x, padded_y


def get_dataloader(path, tokenizer_path, max_len=512, batch_size=32, shuffle=True):
    dataset = GuppyDataset(path, tokenizer_path, max_len)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        collate_fn=collate_fn, num_workers=0, pin_memory=True,
    )


Writing dataset.py


In [6]:
%%writefile train.py
"""GuppyLM training loop."""

import json
import math
import os
import time

import torch

from config import GuppyConfig, TrainConfig
from dataset import get_dataloader
from model import GuppyLM


def get_device(config):
    if config.device == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(config.device)


def get_lr(step, config):
    if step < config.warmup_steps:
        return config.learning_rate * step / config.warmup_steps
    progress = (step - config.warmup_steps) / max(1, config.max_steps - config.warmup_steps)
    coeff = 0.5 * (1 + math.cos(math.pi * progress))
    return config.min_lr + (config.learning_rate - config.min_lr) * coeff


@torch.no_grad()
def evaluate(model, loader, device, max_batches=50):
    model.eval()
    total_loss, n = 0, 0
    for x, y in loader:
        if n >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        total_loss += loss.item()
        n += 1
    model.train()
    return total_loss / max(1, n)


def train():
    mc = GuppyConfig()
    tc = TrainConfig()
    device = get_device(tc)
    torch.manual_seed(tc.seed)

    print(f"Device: {device}")

    tokenizer_path = os.path.join(tc.data_dir, "tokenizer.json")
    model = GuppyLM(mc).to(device)
    print(model.param_summary())

    train_loader = get_dataloader(
        os.path.join(tc.data_dir, "train.jsonl"), tokenizer_path,
        mc.max_seq_len, tc.batch_size, shuffle=True,
    )
    eval_loader = get_dataloader(
        os.path.join(tc.data_dir, "eval.jsonl"), tokenizer_path,
        mc.max_seq_len, tc.batch_size, shuffle=False,
    )
    print(f"Train: {len(train_loader.dataset):,}, Eval: {len(eval_loader.dataset):,}")

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=tc.learning_rate,
        weight_decay=tc.weight_decay, betas=(0.9, 0.95),
    )

    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    os.makedirs(tc.output_dir, exist_ok=True)
    with open(os.path.join(tc.output_dir, "config.json"), "w") as f:
        json.dump({"model": vars(mc), "train": vars(tc)}, f, indent=2)

    model.train()
    step, best_eval = 0, float("inf")
    losses = []
    t0 = time.time()

    print(f"\nTraining for {tc.max_steps} steps...")
    print(f"{'Step':>6} | {'LR':>10} | {'Train':>10} | {'Eval':>10} | {'Time':>8}")
    print("-" * 56)

    while step < tc.max_steps:
        for x, y in train_loader:
            if step >= tc.max_steps:
                break

            x, y = x.to(device), y.to(device)
            lr = get_lr(step, tc)
            for pg in optimizer.param_groups:
                pg["lr"] = lr

            if use_amp:
                with torch.amp.autocast("cuda"):
                    _, loss = model(x, y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), tc.grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                _, loss = model(x, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), tc.grad_clip)
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)
            losses.append(loss.item())

            if step % 100 == 0:
                avg = sum(losses[-100:]) / len(losses[-100:])
                elapsed = time.time() - t0
                print(f"{step:6d} | {lr:10.6f} | {avg:10.4f} | {'--':>10} | {elapsed:7.1f}s")

            if step > 0 and step % tc.eval_interval == 0:
                el = evaluate(model, eval_loader, device)
                avg_train = sum(losses[-tc.eval_interval:]) / min(len(losses), tc.eval_interval)
                elapsed = time.time() - t0
                print(f"{step:6d} | {lr:10.6f} | {avg_train:10.4f} | {el:10.4f} | {elapsed:7.1f}s")

                if el < best_eval:
                    best_eval = el
                    torch.save({
                        "step": step,
                        "model_state_dict": model.state_dict(),
                        "config": vars(mc),
                        "eval_loss": el,
                    }, os.path.join(tc.output_dir, "best_model.pt"))
                    print(f"  -> Best model (eval={el:.4f})")

            if step > 0 and step % tc.save_interval == 0:
                torch.save({
                    "step": step,
                    "model_state_dict": model.state_dict(),
                    "config": vars(mc),
                }, os.path.join(tc.output_dir, f"step_{step}.pt"))

            step += 1

    # Final save
    torch.save({
        "step": step,
        "model_state_dict": model.state_dict(),
        "config": vars(mc),
        "train_losses": losses,
    }, os.path.join(tc.output_dir, "final_model.pt"))

    elapsed = time.time() - t0
    print(f"\nDone! {elapsed:.0f}s, best eval: {best_eval:.4f}")


if __name__ == "__main__":
    train()


Writing train.py


In [7]:
%%writefile inference.py
"""GuppyLM inference — simple chat."""

import json
import time
import uuid

import torch
from tokenizers import Tokenizer

from config import GuppyConfig
from model import GuppyLM


class GuppyInference:
    def __init__(self, checkpoint_path, tokenizer_path, device="cpu"):
        self.device = torch.device(device)
        self.tokenizer = Tokenizer.from_file(tokenizer_path)

        import os
        ckpt = torch.load(checkpoint_path, map_location=self.device, weights_only=False)

        # Load config.json from same directory as the model file
        config_dir = os.path.dirname(os.path.abspath(checkpoint_path))
        config_path = os.path.join(config_dir, "config.json")

        # Extract state_dict — handle both legacy and standard formats
        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            state_dict = ckpt["model_state_dict"]
        else:
            state_dict = ckpt

        # Load config — try config.json first, fall back to embedded config
        if os.path.exists(config_path):
            with open(config_path) as f:
                cfg = json.load(f)
            # Support both HF standard keys and our own keys
            self.config = GuppyConfig(
                vocab_size=cfg.get("vocab_size", 4096),
                max_seq_len=cfg.get("max_position_embeddings", cfg.get("max_seq_len", 128)),
                d_model=cfg.get("hidden_size", cfg.get("d_model", 384)),
                n_layers=cfg.get("num_hidden_layers", cfg.get("n_layers", 6)),
                n_heads=cfg.get("num_attention_heads", cfg.get("n_heads", 6)),
                ffn_hidden=cfg.get("intermediate_size", cfg.get("ffn_hidden", 768)),
                dropout=cfg.get("hidden_dropout_prob", cfg.get("dropout", 0.1)),
                pad_id=cfg.get("pad_token_id", cfg.get("pad_id", 0)),
                bos_id=cfg.get("bos_token_id", cfg.get("bos_id", 1)),
                eos_id=cfg.get("eos_token_id", cfg.get("eos_id", 2)),
            )
        elif isinstance(ckpt, dict) and "config" in ckpt:
            valid_fields = {f.name for f in GuppyConfig.__dataclass_fields__.values()}
            self.config = GuppyConfig(**{k: v for k, v in ckpt["config"].items() if k in valid_fields})
        else:
            print("Warning: No config found, using defaults")
            self.config = GuppyConfig()

        self.model = GuppyLM(self.config).to(self.device)
        filtered = {k: v for k, v in state_dict.items() if k in self.model.state_dict()}
        self.model.load_state_dict(filtered)
        self.model.eval()

        total, _ = self.model.param_count()
        print(f"GuppyLM loaded: {total/1e6:.1f}M params")

    def chat_completion(self, messages, temperature=0.7, max_tokens=64,
                        top_k=50, **kwargs):
        """Chat completion — takes messages, returns response."""
        prompt = self._format_prompt(messages)
        input_ids = self.tokenizer.encode(prompt).ids
        prompt_tokens = len(input_ids)
        input_t = torch.tensor([input_ids], dtype=torch.long, device=self.device)

        output_t, _ = self.model.generate(input_t, max_tokens, temperature, top_k)
        output_text = self.tokenizer.decode(output_t[0].tolist()[prompt_tokens:])
        # Truncate at first <|im_end|> — don't let the model leak into the next turn
        if "<|im_end|>" in output_text:
            output_text = output_text.split("<|im_end|>")[0]
        # Also strip any <|im_start|> fragments
        if "<|im_start|>" in output_text:
            output_text = output_text.split("<|im_start|>")[0]
        resp_text = output_text.strip()

        return {
            "choices": [{
                "message": {"role": "assistant", "content": resp_text},
            }],
        }

    def _format_prompt(self, messages):
        parts = []
        for msg in messages:
            role = msg.get("role", "user")
            content = msg.get("content") or ""
            if role == "system":
                continue
            parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")
        parts.append("<|im_start|>assistant\n")
        return "\n".join(parts)


def main():
    import argparse
    p = argparse.ArgumentParser(description="Chat with Guppy")
    p.add_argument("--checkpoint", default="checkpoints/best_model.pt")
    p.add_argument("--tokenizer", default="data/tokenizer.json")
    p.add_argument("--device", default="cpu")
    args = p.parse_args()

    engine = GuppyInference(args.checkpoint, args.tokenizer, args.device)
    print("\nGuppy Chat (type 'quit' to exit)")
    msgs = []
    while True:
        inp = input("\nYou> ").strip()
        if inp.lower() in ("quit", "exit", "q"):
            break
        msgs.append({"role": "user", "content": inp})
        result = engine.chat_completion(msgs)
        msg = result["choices"][0]["message"]
        if msg.get("content"):
            print(f"Guppy> {msg['content']}")
        msgs.append(msg)


if __name__ == "__main__":
    main()


Writing inference.py


## 3. Prepare Data

Download the fish conversation dataset from HuggingFace and train a BPE tokenizer.

The dataset has 60K single-turn conversations across 60 topics:
greetings, food, temperature, water, tank life, emotions, philosophy (fish-level), and more.

Each sample is formatted as ChatML:
```
<|im_start|>user
hi guppy<|im_end|>
<|im_start|>assistant
hello. the water is nice today.<|im_end|>
```

In [8]:
import json, os
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors

# ── Download from HuggingFace ──
HF_DATASET = 'arman-bd/guppylm-60k-generic'
ds = load_dataset(HF_DATASET)
print(f'Downloaded: {len(ds["train"]):,} train, {len(ds["test"]):,} test samples')

# ── Format into ChatML and save as JSONL ──
os.makedirs('data', exist_ok=True)
texts = []

for split, path in [('train', 'data/train.jsonl'), ('test', 'data/eval.jsonl')]:
    with open(path, 'w') as f:
        for row in ds[split]:
            text = (
                f'<|im_start|>user\n{row["input"]}<|im_end|>\n'
                f'<|im_start|>assistant\n{row["output"]}<|im_end|>'
            )
            f.write(json.dumps({'text': text, 'category': row['category']}) + '\n')
            texts.append(text)
    print(f'  {path}: {len(ds[split]):,} samples')

# ── Train BPE tokenizer on the data ──
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=4096,
    special_tokens=['<pad>', '<|im_start|>', '<|im_end|>'],
    min_frequency=2,
    show_progress=True,
)
tokenizer.train_from_iterator(texts, trainer)
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)
tokenizer.save('data/tokenizer.json')
print(f'  Tokenizer: {tokenizer.get_vocab_size()} tokens')

# ── Preview ──
with open('data/train.jsonl') as f:
    sample = json.loads(f.readline())
print(f'\nSample ({sample["category"]}):\n{sample["text"]}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/557k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/56.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/57000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Downloaded: 57,000 train, 3,000 test samples
  data/train.jsonl: 57,000 samples
  data/eval.jsonl: 3,000 samples
  Tokenizer: 2385 tokens

Sample (snail):
<|im_start|>user
what about a tankmate<|im_end|>
<|im_start|>assistant
would it live along the glass wall. that's my spot.<|im_end|>


## 4. Verify Architecture

Quick sanity check — build the model, print param count, run a dummy forward pass.

In [9]:
from config import GuppyConfig
from model import GuppyLM
import torch

config = GuppyConfig()
model = GuppyLM(config)
print(model.param_summary())
print(f'  Layers: {config.n_layers}, Heads: {config.n_heads}, FFN: {config.ffn_hidden}')
print(f'  Vocab: {config.vocab_size}, Max seq: {config.max_seq_len}')

# Dummy forward pass
x = torch.randint(0, config.vocab_size, (2, 32))
logits, _ = model(x)
print(f'  Forward pass: {x.shape} -> {logits.shape} OK')
del model

GuppyLM: 8,726,016 params (8.7M)
  Layers: 6, Heads: 6, FFN: 768
  Vocab: 4096, Max seq: 128
  Forward pass: torch.Size([2, 32]) -> torch.Size([2, 32, 4096]) OK


## 5. Train

10,000 steps with cosine LR schedule. Takes ~2 min on T4.

The model learns to:
- Respond in short, lowercase sentences
- Stay in character as a fish
- Cover 60 different conversation topics
- Stop generating at the right time (learn the `<|im_end|>` token)

In [10]:
from train import train
train()

Device: cuda
GuppyLM: 8,726,016 params (8.7M)
Train: 57,000, Eval: 3,000

Training for 10000 steps...
  Step |         LR |      Train |       Eval |     Time
--------------------------------------------------------
     0 |   0.000000 |     8.3998 |         -- |     1.1s
   100 |   0.000150 |     5.6982 |         -- |     3.4s
   200 |   0.000300 |     3.2160 |         -- |     6.3s
   200 |   0.000300 |     4.4571 |     2.7572 |     6.8s
  -> Best model (eval=2.7572)
   300 |   0.000300 |     2.5183 |         -- |     9.1s
   400 |   0.000300 |     2.0193 |         -- |    11.4s
   400 |   0.000300 |     2.2688 |     1.7153 |    11.8s
  -> Best model (eval=1.7153)
   500 |   0.000299 |     1.6656 |         -- |    14.2s
   600 |   0.000299 |     1.3678 |         -- |    16.7s
   600 |   0.000299 |     1.5167 |     1.1418 |    17.2s
  -> Best model (eval=1.1418)
   700 |   0.000298 |     1.1415 |         -- |    19.8s
   800 |   0.000298 |     0.9502 |         -- |    22.0s
   800 |  

## 6. Test

Chat with the trained model. Each message is independent (single-turn).

In [11]:
from inference import GuppyInference
import torch

engine = GuppyInference(
    'checkpoints/best_model.pt', 'data/tokenizer.json',
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

def chat(prompt):
    r = engine.chat_completion([{'role': 'user', 'content': prompt}], max_tokens=64)
    return r['choices'][0]['message'].get('content', '').strip()

# Test across different topics
tests = [
    ('hi guppy',                      'greeting'),
    ('are you hungry',                'food'),
    ('it is really hot today',        'temperature'),
    ('how is the water',              'water'),
    ('do you like bubbles',           'bubbles'),
    ('what is the internet',          'confused'),
    ('do you get lonely',             'lonely'),
    ('the cat is looking at you',     'cat'),
    ('tell me a joke',                'joke'),
    ('what do you dream about',       'dreams'),
    ('do you love me',                'love'),
    ('what is the meaning of life',   'meaning'),
    ('sorry i tapped the glass',      'glass_tap'),
    ('it is raining outside',         'rain'),
    ('goodnight guppy',               'night'),
]

print(f'{"Topic":<12s}  {"You":<35s}  Guppy')
print('=' * 100)
for prompt, topic in tests:
    reply = chat(prompt)
    print(f'{topic:<12s}  {prompt:<35s}  {reply[:128]}')


GuppyLM loaded: 8.7M params
Topic         You                                  Guppy
greeting      hi guppy                             hey. you look big today. you always look big. it's a good day to be a fish.
food          are you hungry                       is it food time. please say yes. i ate earlier but i forgot already. i promise to eat all of it.
temperature   it is really hot today               warm water is bad for me. i need cool water to be ok.
water         how is the water                     i can taste the water. it tastes still. i can breathe easy.
bubbles       do you like bubbles                  bubbles. i think they're made of air. whatever that is.
confused      what is the internet                 the internet. that sounds like a human thing. i am a fish.
lonely        do you get lonely                    i talk to the plant sometimes. it doesn't answer. i wouldn't say no to a snail though. but a snail would be nice.
cat           the cat is looking at you   

## 7. Export & Upload to HuggingFace

Export the model in both PyTorch and ONNX (quantized uint8, ~9 MB) formats,
then upload everything to HuggingFace in one go.

Set your token and repo below.

In [12]:
!pip install -q onnx onnxruntime onnxscript

from huggingface_hub import HfApi, login
import torch, json, os, shutil
from config import GuppyConfig
from model import GuppyLM

HF_TOKEN = os.environ.get('HF_TOKEN', '')  # Or paste your token here
HF_REPO = os.environ.get('HF_REPO', 'arman-bd/guppylm-9M')  # Or change this

# Load checkpoint
ckpt = torch.load('checkpoints/best_model.pt', map_location='cpu', weights_only=False)
cfg = ckpt['config']
os.makedirs('hf_export', exist_ok=True)

# ── PyTorch format ──
torch.save(ckpt['model_state_dict'], 'hf_export/pytorch_model.bin')

with open('hf_export/config.json', 'w') as f:
    json.dump({
        'model_type': 'guppylm',
        'architectures': ['GuppyLM'],
        'vocab_size': cfg['vocab_size'],
        'max_position_embeddings': cfg['max_seq_len'],
        'hidden_size': cfg['d_model'],
        'num_hidden_layers': cfg['n_layers'],
        'num_attention_heads': cfg['n_heads'],
        'intermediate_size': cfg['ffn_hidden'],
        'hidden_dropout_prob': cfg.get('dropout', 0.1),
        'pad_token_id': cfg['pad_id'],
        'bos_token_id': cfg['bos_id'],
        'eos_token_id': cfg['eos_id'],
    }, f, indent=2)

shutil.copy('data/tokenizer.json', 'hf_export/tokenizer.json')
print(f'pytorch_model.bin: {os.path.getsize("hf_export/pytorch_model.bin")/1e6:.1f} MB')

# ── ONNX format (quantized uint8) ──
valid_fields = {f.name for f in GuppyConfig.__dataclass_fields__.values()}
config = GuppyConfig(**{k: v for k, v in cfg.items() if k in valid_fields})
model = GuppyLM(config)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

dummy = torch.randint(0, config.vocab_size, (1, 32))
fp32_path = 'hf_export/model_fp32.onnx'
torch.onnx.export(
    model, (dummy,), fp32_path,
    input_names=['input_ids'], output_names=['logits'],
    dynamic_axes={'input_ids': {0: 'batch', 1: 'seq_len'},
                  'logits': {0: 'batch', 1: 'seq_len'}},
    opset_version=17,
)

from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(fp32_path, 'hf_export/model.onnx', weight_type=QuantType.QUInt8)
os.remove(fp32_path)
print(f'model.onnx:       {os.path.getsize("hf_export/model.onnx")/1e6:.1f} MB (uint8)')

# ── Upload ──
if HF_TOKEN:
    login(token=HF_TOKEN)
    api = HfApi()
    api.create_repo(HF_REPO, exist_ok=True)
    api.upload_folder(folder_path='hf_export', repo_id=HF_REPO, repo_type='model')
    print(f'Done! https://huggingface.co/{HF_REPO}')
else:
    print('No HF_TOKEN — exported locally to hf_export/')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 18.7 MB/s eta 0:00:00
pytorch_model.bin: 34.9 MB


/tmp/ipykernel_3000/2238562038.py:47: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0506 16:23:35.997000 3000 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `GuppyLM([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `GuppyLM([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
W0506 16:23:43.245000 3000 torch/onnx/_internal/exporter/_core.py:749] Output node output has None output. The output is ignored in the exported graph. Please ensure the graph output order is expected
W0506 16:23:43.248000 3000 torch/onnx/_internal/exporter/_core.py:1193] Skipping constant argument ConstantArgument(name='', value=None)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


model.onnx:       8.8 MB (uint8)
No HF_TOKEN — exported locally to hf_export/


## 8. Download

Or download the model locally as a tar.gz.

In [13]:
import os

!cd /content && tar czf guppylm.tar.gz \
    guppy/checkpoints/best_model.pt \
    guppy/checkpoints/config.json \
    guppy/data/tokenizer.json \
    guppy/model.py \
    guppy/config.py \
    guppy/inference.py \
    guppy/hf_export/model.onnx

sz = os.path.getsize('/content/guppylm.tar.gz') / 1e6
print(f'Package: /content/guppylm.tar.gz ({sz:.1f} MB)')

try:
    from google.colab import files
    files.download('/content/guppylm.tar.gz')
except ImportError:
    print('Not in Colab — download manually from the file browser.')

Package: /content/guppylm.tar.gz (39.9 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>